<a href="https://colab.research.google.com/github/cbonnin88/mes_allocs-ETL/blob/main/Product_Manager_ToolKit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install streamlit pyngrok joblib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 56.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 93.3 MB/s eta 0:00:00


In [22]:
%%writefile app.py
import streamlit as st
import polars as pl
import joblib
from google.cloud import bigquery
from google.colab import auth
import json

# 1. Page Configuration & Logo
st.set_page_config(page_title='Mes-Allocs PM Toolkit',page_icon="📈")
st.image('/content/Gemini_Generated_Image_ipwd7vipwd7vipwd.png',width=150)
st.title('🛡️ Mes-Allocs PM Command Center')

# 2. Load the Phase 5 ML Model
@ st.cache_resource
def load_assets():
  model_path = '/content/model.pkl'
  return joblib.load(model_path)

model = load_assets()

# 3. Sidebar: User Investigation
st.sidebar.header('User Search')
user_input = st.sidebar.text_input('Enter User ID',placeholder='user_0')

if user_input:
  service_account_path = '/content/mes-allocs-analysis.json'
  # Fetch data from the dbt mart in Bigquery
  client = bigquery.Client.from_service_account_json(service_account_path)
  query = f"""
    SELECT * FROM `mes-allocs-analysis.dbt_mes_allocs.fct_product_events`
    WHERE user_id = '{user_input}'
  """

  df_product = client.query(query).to_dataframe()

  if not df_product.empty:
    # Display Core Metrics
    st.subheader(f'Insights for {user_input}')
    col1,col2,col3 = st.columns(3)

    total_activity = len(df_product)
    plan = df_product['subscription_plan'].iloc[0]
    status = df_product['user_status'].iloc[0]

    col1.metric('Total Events',total_activity)
    col2.metric('Status',status)
    col3.metric('Plan',plan.capitalize())

    # 4. Churn Risk (ML Implementation)
    # Using the logic from Phase 5 Feature Engineering
    sim_start = len(df_product[df_product['event_type']== 'simulation_started'])
    price = df_product['plan_price'].iloc[0]

    # Formatting features for the model
    features = [[total_activity, price, sim_start]]
    churn_proba = model.predict_proba(features)[0][0]

    st.markdown('---')
    st.write('### 🚨 Predictive Churn Risk')

    if churn_proba > 0.7:
      st.error(f'**High Risk:** {churn_proba:.1%} probability of churn')
      st.button('Generate Retention Disount Code')
    elif churn_proba > 0.4:
      st.warning(f'**Medium Risk** {churn_proba:.1%} probability of churn')
    else:
      st.success(f'**Low Risk:** {churn_proba:.1%} probability of churn')

    # 5. Amplitude-style Behavioral Timeline
    st.write('### Recent Activity Timeline')
    st.table(df_product[['event_at','event_type','device_type']].sort_values('event_at',ascending=False).head(5))

else:
  st.error('User ID not found in warehouse')

Overwriting app.py


In [23]:
from pyngrok import ngrok
from google.colab import userdata

ngrok_auth_token_key = 'ngrok_token'
ngrok_token = userdata.get('ngrok_token')

if ngrok_token:
    ngrok.set_auth_token(ngrok_token)
else:
    print(f"Warning: ngrok auth token not found in Colab secrets. Please add it using the key '{ngrok_auth_token_key}'.")
    print("You can get your token from https://dashboard.ngrok.com/auth/your-authtoken")
    print("Without the token, ngrok will not be able to create a tunnel.")


# Connect to Streamlit port
public_url = ngrok.connect(8501)
print(f"Click here to open your App: {public_url}")

# Run Streamlit
!streamlit run app.py --server.port 8501

Click here to open your App: NgrokTunnel: "https://bc6a-34-69-102-193.ngrok-free.app" -> "http://localhost:8501"



  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.69.102.193:8501

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature 

  Stopping...
  Stopping...
